# Payphone: merged HF → GGUF on Kaggle

Use this **after** [kaggle_full_import.ipynb](https://github.com/s4ifk99/ProjectPayphone/blob/main/notebooks/kaggle_full_import.ipynb) (or with your own merged checkpoint). It clones **llama.cpp**, runs **`convert_hf_to_gguf.py`**, and lets you download **`payphone-story.gguf`** for Ollama.

**Setup**

1. [kaggle.com](https://kaggle.com) — **Settings** → **Internet** ON (needed for `git clone` + `pip`).
2. **Accelerator:** GPU is optional; conversion is **CPU + RAM** heavy. If the kernel dies, try **Settings → Accelerator → GPU** (more RAM) or change **`OUTTYPE`** below to **`q4_k_m`**.
3. **Add data** → attach a dataset that contains either:
   - **`payphone-merged-hf.zip`** (from the merge notebook), or
   - the **unzipped** merged folder (must contain **`config.json`** next to **`model*.safetensors`**).
4. Run all cells.

**Note:** If conversion OOMs, set **`OUTTYPE = "q4_k_m"`** in the convert cell and re-run from that cell.

**NF4 / `absmax` error:** Older merge zips saved **bitsandbytes NF4** tensors (`*.weight.absmax`). Section **2b** reloads and **`dequantize()`**s to dense FP16 for **llama.cpp** — **GPU recommended** (bitsandbytes + 7B).

**Pip “dependency conflicts”:** Section 1 often ends with **`ERROR: pip's dependency resolver`** and a long list (numpy, protobuf, `google-colab`, etc.). That is a **warning** about Kaggle’s pre-installed packages, not this notebook failing. **If the cell prints `OK: llama.cpp at /kaggle/working/llama.cpp`, continue** to the next cell.

**`ResolutionImpossible` / `torch~=2.6.0`:** The notebook no longer runs **`pip install -r llama.cpp/requirements.txt`** (that file pins **torch** and fights Kaggle’s **`-c` torch**). GGUF conversion uses **`gguf-py`** inside the clone.

**If the next cell “does nothing”:** Wait until cell 1 shows a number (not `[*]`) in the margin — pip can run several minutes. **Do not** use indented `!shell` lines on Kaggle; this notebook uses `subprocess` instead. If the UI freezes from huge pip logs, **Session → Restart**, run cell 1 again, then continue.

**`numpy.dtype size changed` / `mtrand`:** Kaggle’s image mixed numpy wheels after pip. Cell 1 reinstalls **numpy/scipy** before and after other installs. If section **2b** still fails, **Session → Restart** and **Run All** from cell 1.


## 1. Clone llama.cpp and install Python deps


In [ ]:
import os
import subprocess
import sys

WORKING = "/kaggle/working"
LLAMA = os.path.join(WORKING, "llama.cpp")

print("Cell 1: installing packages + cloning llama.cpp (may take several minutes)...")


def _pip_install(args):
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *args],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        r.check_returncode()


# Align numpy/scipy wheels BEFORE importing torch/transformers (fixes
# "numpy.dtype size changed" / mtrand binary mismatch after pip churn on Kaggle).
_pip_install(
    [
        "--force-reinstall",
        "numpy>=1.26,<2.1",
        "scipy>=1.11",
        "scikit-learn>=1.5.0",
    ]
)

import torch

with open("/tmp/constraints.txt", "w") as f:
    f.write(f"torch=={torch.__version__}\n")
_pip_install(
    [
        "-c",
        "/tmp/constraints.txt",
        "transformers>=4.45",
        "accelerate",
        "bitsandbytes",
        "safetensors",
    ]
)

if not os.path.isdir(os.path.join(LLAMA, ".git")):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/ggerganov/llama.cpp",
            LLAMA,
        ],
        check=True,
    )
else:
    print("Reusing existing", LLAMA)

# Do NOT pip install llama.cpp/requirements.txt: it pulls torch~=2.6.0 from PyTorch
# index and hits ResolutionImpossible with -c torch=={Kaggle}. convert_hf_to_gguf.py
# imports gguf from the cloned repo (gguf-py/) and only needs torch/numpy/transformers.
_pip_install(["-c", "/tmp/constraints.txt", "sentencepiece", "protobuf"])

# Re-pin numpy/scipy after other installs (they may have pulled a conflicting numpy).
_pip_install(
    [
        "-c",
        "/tmp/constraints.txt",
        "--force-reinstall",
        "numpy>=1.26,<2.1",
        "scipy>=1.11",
        "scikit-learn>=1.5.0",
    ]
)

print("OK: llama.cpp at", LLAMA)
print("Section 1 done — run the next cell (locate merged HF).")
print(
    "If section 2b still raises numpy.dtype / mtrand errors: Session → Restart, "
    "then Run All from this cell."
)


## 2. Locate merged Hugging Face folder (from Add data)

Scans **`/kaggle/input`** for **`config.json`** plus weight files; if only a **`.zip`** is present, extracts under **`/kaggle/working/unzipped_hf`**.


In [ ]:
import os
import zipfile

INPUT_ROOT = "/kaggle/input"
WORKING = "/kaggle/working"
os.makedirs(WORKING, exist_ok=True)

if not os.path.isdir(INPUT_ROOT):
    raise FileNotFoundError(
        "No /kaggle/input — add your dataset (payphone-merged-hf.zip) via Add data on the right."
    )


def _has_weights(path):
    names = os.listdir(path)
    if any(n == "model.safetensors" for n in names):
        return True
    if any(n.startswith("model-") and n.endswith(".safetensors") and "of-" in n for n in names):
        return True
    if "model.safetensors.index.json" in names:
        return True
    return False


def _is_merged_hf(path):
    return os.path.isfile(os.path.join(path, "config.json")) and _has_weights(path)


merged_dir = None
for root, _, files in os.walk(INPUT_ROOT):
    if _is_merged_hf(root):
        merged_dir = root
        print("Found merged HF at:", merged_dir)
        break

if not merged_dir:
    print("No merged folder under input; trying .zip under /kaggle/input ...")
    for name in sorted(os.listdir(INPUT_ROOT)):
        d = os.path.join(INPUT_ROOT, name)
        if not os.path.isdir(d):
            continue
        for f in os.listdir(d):
            if not f.endswith(".zip"):
                continue
            zip_path = os.path.join(d, f)
            extract_dir = os.path.join(WORKING, "unzipped_hf")
            os.makedirs(extract_dir, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(extract_dir)
            for root, _, _ in os.walk(extract_dir):
                if _is_merged_hf(root):
                    merged_dir = root
                    print("Found merged HF after unzip:", merged_dir)
                    break
            if merged_dir:
                break
        if merged_dir:
            break

if not merged_dir:
    raise FileNotFoundError(
        "Add a dataset with payphone-merged-hf.zip or the unzipped folder "
        "(config.json + model*.safetensors next to it)."
    )


## 2b. NF4 → dense FP16 (if `convert_hf_to_gguf` failed on `*.absmax`)

Merge-only exports can save **bitsandbytes NF4** sidecars (`model.layers.*.weight.absmax`). **llama.cpp** cannot map those.

This cell **first scans** `merged_dir` for `.absmax` / `weight_scale_inv` **without** importing `transformers`. If none are found (dense FP16 merge), it **skips** torch/transformers entirely — use that path when you can.

If NF4 tensors are present, dequant runs in a **separate Python subprocess** (not the notebook kernel) so **numpy / transformers** load cleanly — this avoids the common **`numpy.dtype size changed`** / **`mtrand`** crash in Jupyter. **GPU** required.

If your zip was built with an **updated** `kaggle_full_import.ipynb` (dequantize before save), section 2b only prints a short skip message.

In [ ]:
import glob
import json
import os
import shutil

WORKING = "/kaggle/working"
DENSE_DIR = os.path.join(WORKING, "merged_dense_fp16")
OFFLOAD_DIR = os.path.join(WORKING, "offload_dense")


def _has_nf4_aux_keys(merged_path):
    idx = os.path.join(merged_path, "model.safetensors.index.json")
    if os.path.isfile(idx):
        with open(idx) as f:
            wm = json.load(f).get("weight_map", {})
        return any(".absmax" in k or "weight_scale_inv" in k for k in wm)
    st = glob.glob(os.path.join(merged_path, "*.safetensors"))
    if not st:
        return False
    from safetensors import safe_open

    with safe_open(st[0], framework="pt") as f:
        keys = list(f.keys())
    return any(".absmax" in k or "weight_scale_inv" in k for k in keys)


def export_dense_fp16_if_needed(src_dir, out_dir):
    if not _has_nf4_aux_keys(src_dir):
        print("No NF4 aux tensors (e.g. .absmax); using merged folder as-is for GGUF.")
        print("(Skipped torch/transformers — avoids import errors on fragile Kaggle stacks.)")
        return src_dir

    # Run dequant in a fresh subprocess: Jupyter keeps stale numpy in sys.modules after pip;
    # transformers→masking_utils→torch._dynamo then hits mtrand ABI errors in-kernel.
    import subprocess
    import sys

    print(
        "NF4 tensors detected — dequantizing in subprocess (fresh imports; avoids kernel numpy bugs)..."
    )
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(OFFLOAD_DIR, exist_ok=True)

    script = os.path.join(WORKING, "dequant_nf4.py")
    with open(script, "w", encoding="utf-8") as f:
        f.write(
            '''#!/usr/bin/env python3
import gc
import json
import os
import sys

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


def main():
    src, out, offload = sys.argv[1], sys.argv[2], sys.argv[3]
    if not torch.cuda.is_available():
        print("ERROR: GPU required for NF4 dequantize", file=sys.stderr)
        sys.exit(2)
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        src,
        quantization_config=bnb_cfg,
        device_map="auto",
        max_memory={0: "14GiB", "cpu": "60GiB"},
        offload_folder=offload,
        trust_remote_code=True,
    )
    tok = AutoTokenizer.from_pretrained(src, trust_remote_code=True)
    model = model.dequantize()
    model.save_pretrained(out, safe_serialization=True)
    tok.save_pretrained(out)
    cfg_path = os.path.join(out, "config.json")
    with open(cfg_path) as f:
        cfg = json.load(f)
    for k in ("quantization_config", "pre_quantization_dtype", "_pre_quantization_dtype"):
        cfg.pop(k, None)
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print("OK:", out)


if __name__ == "__main__":
    main()
'''
        )
    subprocess.run([sys.executable, script, src_dir, out_dir, OFFLOAD_DIR], check=True)
    print("Dense FP16 checkpoint:", out_dir)
    return out_dir


merged_for_gguf = export_dense_fp16_if_needed(merged_dir, DENSE_DIR)

## 3. Convert to GGUF


In [ ]:
import os
import subprocess
import sys

WORKING = "/kaggle/working"
LLAMA = os.path.join(WORKING, "llama.cpp")
OUT_GGUF = os.path.join(WORKING, "payphone-story.gguf")

# q8_0 matches scripts/convert_to_gguf.sh; if the kernel dies (OOM), try "q4_k_m".
OUTTYPE = "q8_0"

script = os.path.join(LLAMA, "convert_hf_to_gguf.py")
assert os.path.isfile(script), "Run section 1 first (clone llama.cpp)."

subprocess.run(
    [sys.executable, script, merged_for_gguf, "--outfile", OUT_GGUF, "--outtype", OUTTYPE],
    check=True,
)
print("Wrote:", OUT_GGUF, "size bytes:", os.path.getsize(OUT_GGUF) if os.path.isfile(OUT_GGUF) else "missing")


## 4. Download GGUF


In [ ]:
import os
from IPython.display import FileLink, display

WORKING = "/kaggle/working"
z = os.path.join(WORKING, "payphone-story.gguf")
if os.path.isfile(z):
    os.chdir(WORKING)
    print("Download GGUF for Ollama:")
    display(FileLink("payphone-story.gguf", result_html_prefix="Download: "))
else:
    print("ERROR: payphone-story.gguf not found. Re-run the convert cell.")
